# Fresh HotpotQA SDPO training in Colab

Before running, choose **Runtime → Disconnect and delete runtime**, reconnect, and select an **L4 GPU**. The HotpotQA JSONL files and source configuration are expected to already exist in Google Drive. Run every cell in order.

In [ ]:
import os
import sys
from urllib.parse import urlunparse

from google.colab import drive, userdata

drive.mount("/content/drive")

hf_token = userdata.get("HF_TOKEN")
hf_infer = userdata.get("HF_INFER")
assert hf_token, "HF_TOKEN is missing from Colab Secrets"
assert hf_infer, "HF_INFER is missing from Colab Secrets"

os.environ["HF_TOKEN"] = hf_token
os.environ["OPENAI_API_KEY"] = hf_infer
os.environ["OPENAI_BASE_URL"] = urlunparse(
    ("https", "router.huggingface.co", "/v1", "", "", "")
)
os.environ["PYTHONUNBUFFERED"] = "1"

assert os.environ["OPENAI_BASE_URL"] == "https://router.huggingface.co/v1"
assert "[" not in os.environ["OPENAI_BASE_URL"]
assert "]" not in os.environ["OPENAI_BASE_URL"]

import torch

assert torch.cuda.is_available(), "No GPU detected"
assert torch.cuda.is_bf16_supported(), "Select a bf16-capable GPU such as L4 or A100"

print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Actual base URL:", repr(os.environ["OPENAI_BASE_URL"]))

## Clone the pinned repository and install dependencies

In [ ]:
%cd /content
!git clone https://github.com/mmzinn12/rlm-ib.git
%cd /content/rlm-ib
!git checkout 18cf8b7d7b78a088eb1383f005f1e1cd6f4f165e
!git rev-parse HEAD

In [ ]:
%pip install -q -e /content/rlm-ib -e "/content/rlm-ib/training[colab,hub-datasets]"

In [ ]:
import importlib.metadata as metadata
import openai
import rlm
import rlm_train

print("OpenAI:", metadata.version("openai"))
print("RLM:", metadata.version("rlms"))
print("RLM Train:", metadata.version("rlm-train"))

## Verify package sanity checks

Run the SDPO package tests (including teacher-target normalization) and print the results.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_sdpo.py", "-q"],
    cwd="/content/rlm-ib/training",
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Package SDPO sanity checks failed"
print("Package SDPO sanity checks passed.")


## Create a fresh BF16 run configuration

The cell preserves older runs and automatically chooses the next unused run number.

In [ ]:
import json
from pathlib import Path

SOURCE_CONFIG = Path(
    "/content/drive/MyDrive/rlm-ib-configs/"
    "hotpotqa-dense-qwen25math15b-sdpo-qwen25-7b-judge-v1-edge-v3.json"
)
assert SOURCE_CONFIG.exists(), f"Missing source configuration: {SOURCE_CONFIG}"

config = json.loads(SOURCE_CONFIG.read_text())
config["model"]["precision"] = "bf16"
config["optimization"]["learning_rate"] = 0.00005
config["optimization"]["max_gradient_norm"] = 1.0
config["optimization"]["warmup_steps"] = 5
config["optimization"]["max_optimizer_steps"] = 25
config["optimization"]["gradient_accumulation_steps"] = 2
config["optimization"]["policy_weight"] = 0.0
config["optimization"]["sdpo_weight"] = 1.0
config["optimization"]["gram_weight"] = 0.0
config["judge"]["prompt_schema_version"] = "trajectory-feedback-v3"
config["judge"]["mode"] = "categorical"  # Change to "full" if desired.

drive_root = Path(config["output"]["google_drive_root"])
output_parent = drive_root / config["output"]["output_directory"]
run_base = (
    "hotpotqa-dense-qwen25math15b-"
    "sdpo-qwen25-7b-judge-bf16-fresh"
)

run_number = 1
while (output_parent / f"{run_base}-{run_number:02d}").exists():
    run_number += 1

fresh_run_name = f"{run_base}-{run_number:02d}"
config["output"]["run_name"] = fresh_run_name

FRESH_CONFIG = SOURCE_CONFIG.parent / f"{fresh_run_name}.json"
FRESH_RUN_DIRECTORY = output_parent / fresh_run_name
train_path = Path(config["dataset"]["path"])
eval_path = Path(
    config["experiment"]["evaluation"]["benchmarks"][0]["path"]
)

assert train_path.exists(), f"Missing training data: {train_path}"
assert eval_path.exists(), f"Missing evaluation data: {eval_path}"
assert not FRESH_RUN_DIRECTORY.exists()

FRESH_CONFIG.write_text(json.dumps(config, indent=2) + "\n")

print("Fresh configuration:", FRESH_CONFIG)
print("Fresh run directory:", FRESH_RUN_DIRECTORY)
print("Precision:", config["model"]["precision"])
print("Learning rate:", config["optimization"]["learning_rate"])
print("Train rows:", sum(1 for _ in train_path.open()))
print("Evaluation rows:", sum(1 for _ in eval_path.open()))

## Launch training in the background

In [ ]:
import subprocess
import sys
from pathlib import Path

LOG_PATH = Path("/content/rlm-train-live.log")
assert os.environ["OPENAI_BASE_URL"] == "https://router.huggingface.co/v1"

log_stream = LOG_PATH.open("w")
training_process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "rlm_train.colab.launcher",
        str(FRESH_CONFIG),
    ],
    cwd="/content/rlm-ib",
    env=os.environ.copy(),
    stdout=log_stream,
    stderr=subprocess.STDOUT,
)
log_stream.close()

print("Training launched.")
print("PID:", training_process.pid)
print("Configuration:", FRESH_CONFIG)
print("Run directory:", FRESH_RUN_DIRECTORY)
print("Live log:", LOG_PATH)

## Monitor training

Run this cell repeatedly. The first decisive success signal is **Optimizer steps completed: 1**.

In [ ]:
import json
import subprocess

exit_code = training_process.poll()
if exit_code is None:
    print("PROCESS: RUNNING — PID", training_process.pid)
elif exit_code == 0:
    print("PROCESS: COMPLETED SUCCESSFULLY")
else:
    print("PROCESS: FAILED — exit code", exit_code)

print("\nGPU:")
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.used,memory.total,utilization.gpu",
        "--format=csv,noheader",
    ],
    check=False,
)

teacher_directory = FRESH_RUN_DIRECTORY / "teacher-targets"
teacher_targets = list(teacher_directory.glob("*.json"))
metrics_path = FRESH_RUN_DIRECTORY / "metrics.jsonl"
summary_path = FRESH_RUN_DIRECTORY / "summary.json"
checkpoint_path = FRESH_RUN_DIRECTORY / "latest.json"

print("\nTeacher targets completed:", len(teacher_targets))
if metrics_path.exists():
    metrics = metrics_path.read_text().splitlines()
    print("Optimizer steps completed:", len(metrics))
    if metrics:
        print("\nLatest metrics:")
        print(json.dumps(json.loads(metrics[-1]), indent=2))
else:
    print("Optimizer steps completed: 0")

print("Checkpoint exists:", checkpoint_path.exists())
print("Final summary exists:", summary_path.exists())

if LOG_PATH.exists():
    print("\nLATEST LOG:")
    print("\n".join(LOG_PATH.read_text(errors="replace").splitlines()[-80:]))